In [ ]:
import torch
import faiss
import numpy as np
from groq import Groq
from transformers import AutoTokenizer, AutoModel

# -------- CONFIG --------
GROQ_API_KEY = "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"
MODEL_NAME = "llama3-70b-8192"

client = Groq(api_key=GROQ_API_KEY)

# Load MiniLM model for embeddings
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

# -------- Embed function --------
def embed(texts):
    tokens = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**tokens)
    embeddings = model_output.last_hidden_state.mean(dim=1).numpy()
    return embeddings

# -------- Your documents --------
docs = [
    "The sun rises in the east and sets in the west.",
    "Python is a popular programming language known for its simplicity.",
    "Groq provides ultra-fast inference for LLMs like LLaMA 3 and Mixtral.",
    "The capital of France is Paris.",
    "LLaMA 3 is an open-weight LLM developed by Meta."
]

# -------- Index documents --------
doc_embeddings = embed(docs)
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(np.array(doc_embeddings))

# -------- RAG Query --------
def rag_query(user_question, top_k=2):
    question_embedding = embed([user_question])
    D, I = index.search(question_embedding, top_k)
    retrieved_docs = [docs[i] for i in I[0]]

    context = "\n".join(retrieved_docs)
    prompt = f"""Answer the question using the context below:

Context:
{context}

Question:
{user_question}

Answer:"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content.strip()

# -------- Run it! --------
if __name__ == "__main__":
    while True:
        user_input = input("Ask a question (or type 'exit'): ")
        if user_input.lower() == 'exit':
            break
        answer = rag_query(user_input)
        print("\nAnswer:", answer, "\n")


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

C:\Users\Sunith\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sunith\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...


Answer: I think there might be a misunderstanding in the question! France is a country, and it has a capital city, which is Paris. So, the question "which country's capital city is France" doesn't quite make sense.

If you meant to ask "What is the capital of France?", the answer would be Paris! 



Ask a question (or type 'exit'):  which country's capital city is paris?



Answer: The answer is: France. 



Ask a question (or type 'exit'):  who is goddess durga?



Answer: I think there might be some confusion here! The context provided doesn't seem to relate to the question about Goddess Durga.

However, I'd be happy to help answer your question! Goddess Durga is a major deity in Hinduism, worshipped as a symbol of strength, protection, and femininity. She is often depicted as a powerful, fearless warrior goddess, riding a lion or a tiger, and is believed to be the destroyer of evil and the embodiment of feminine power. 

